# 03. Model and Cost Comparison

**Purpose:** Compare practical trade-offs between models (cost/speed vs accuracy) by running the same prompt on two models.

**Models:**
- Baseline: **gemini-2.5-flash**
- Advanced: **gemini-2.5-pro**

**Key Takeaway:** Model selection is a business decision balancing accuracy, latency, and cost.


## Step 1: Install Dependencies

In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r esmt-workshop/requirements.txt

## Step 2: Load Shared Utilities

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/esmt-workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address

print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Configure the API Client

In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ.get('WORKSHOP_EMAIL', '')

# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')
print('STUDENT_EMAIL =', STUDENT_EMAIL)


## Step 4: Limited-Run Model Comparison (edit **one cell only**)

Use the next cell to compare models quickly on a small subset.

**Do this in order:**
1. Paste your best prompt from Notebook 02 into `STUDENT_PROMPT_TEMPLATE`.
2. **Run the baseline model first**:
   - Set `MODEL_NAME = FLASH_MODEL`
   - Set `STAGE_NAME = 'baseline'`
   - Run the cell → record `micro_accuracy` + `Runtime (sec)`.
3. **Run the advanced model second**:
   - Set `MODEL_NAME = PRO_MODEL`
   - Set `STAGE_NAME = 'advanced'`
   - Run the same cell again → record `micro_accuracy` + `Runtime (sec)`.

**Rule:** Only edit:
- `STUDENT_PROMPT_TEMPLATE` / `PROMPT_TO_RUN`
- `MODEL_NAME`
- `STAGE_NAME`


In [ ]:
import time

# -----------------------------
# PROMPT (paste your best prompt from Notebook 02)
# -----------------------------
STUDENT_PROMPT_TEMPLATE = '''You are a highly precise AML address parsing system.

Input address:
{address}

Return STRICT JSON only using this exact schema:
{schema}

Rules:
- Town Name: city/locality only (no street/state/province).
- Postal Code: postal token(s) only (preserve formatting).
- Remaining Address: everything else (street, number, unit, state/province, etc.).
- Country Code: ISO alpha-2 uppercase (convert country names when explicit).
- Do not invent values; use "" when unknown.
'''

PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE


# -----------------------------
# MODEL + STAGE (students change these)
# -----------------------------
FLASH_MODEL = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
PRO_MODEL = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

# Run baseline first:
MODEL_NAME = FLASH_MODEL
STAGE_NAME = 'baseline'   # must be one of: baseline, prompt_tuned, advanced, two_stage

# Then switch to:
# MODEL_NAME = PRO_MODEL
# STAGE_NAME = 'advanced'


# -----------------------------
# FIXED PARAMS (from original notebooks)
# -----------------------------
TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
MAX_TOKENS = None
MAX_WORKERS = 8

USE_GUARDRAILS = False  # default for this notebook
MOCK_MODE = True        # set False when ready for real gateway


# -----------------------------
# DATA (limited run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
dev_small = dev_df.head(10).copy()


# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_small,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
    mock_mode=MOCK_MODE,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

# Generate report (same output style as original notebooks)
report = evaluate_predictions(pred_df, dev_small, email=STUDENT_EMAIL)

print(report['summary'])
display(report['field_metrics'])


## Step 6: Full Dataset Run (your **publishable** result)

Choose **one** model to run on the full dev set. This score is what you will publish later.

**Pick ONE configuration and run this cell once:**

- **Baseline publish run**:
  - `MODEL_NAME = FLASH_MODEL`
  - `STAGE_NAME = 'baseline'`

- **Advanced publish run**:
  - `MODEL_NAME = PRO_MODEL`
  - `STAGE_NAME = 'advanced'`

**Rule:** Only edit:
- `STUDENT_PROMPT_TEMPLATE` / `PROMPT_TO_RUN`
- `MODEL_NAME`
- `STAGE_NAME`


In [ ]:
import time

# -----------------------------
# PROMPT (paste the SAME best prompt you used above)
# -----------------------------
STUDENT_PROMPT_TEMPLATE = '''You are a highly precise AML address parsing system.

Input address:
{address}

Return STRICT JSON only using this exact schema:
{schema}

Rules:
- Town Name: city/locality only (no street/state/province).
- Postal Code: postal token(s) only (preserve formatting).
- Remaining Address: everything else (street, number, unit, state/province, etc.).
- Country Code: ISO alpha-2 uppercase (convert country names when explicit).
- Do not invent values; use "" when unknown.
'''

PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE


# -----------------------------
# MODEL + STAGE (students choose ONE for publishable result)
# -----------------------------
FLASH_MODEL = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
PRO_MODEL = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

# Choose ONE:
MODEL_NAME = FLASH_MODEL
STAGE_NAME = 'baseline'

# Or:
# MODEL_NAME = PRO_MODEL
# STAGE_NAME = 'advanced'


# -----------------------------
# FIXED PARAMS (from original notebooks)
# -----------------------------
TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
MAX_TOKENS = None
MAX_WORKERS = 8

USE_GUARDRAILS = False
MOCK_MODE = True  # set False when ready for real gateway


# -----------------------------
# DATA (full run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')


# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
    mock_mode=MOCK_MODE,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

report = evaluate_predictions(pred_df, dev_df, email=STUDENT_EMAIL)

print(report['summary'])
display(report['field_metrics'])


## Step 7: Publish Placeholder

Later, you will publish the results from **Step 6**. For now, this is a placeholder.

- Save predictions/report artifacts
- Submit score via the workshop publishing mechanism (to be provided)


In [ ]:
# Placeholder (do not run)
PUBLISH_TO_LEADERBOARD = False

if PUBLISH_TO_LEADERBOARD:
    print('Publishing is not enabled in this notebook yet.')
